# Colab bootstrap — crop-disease-xai

Run top to bottom at the start of every Colab session.

**One-time setup**
- Colab → *Secrets* (key icon): add `WANDB_API_KEY`, and `GITHUB_TOKEN` if the repo is private.
- Put `plantvillage.zip`, `plantdoc.zip` (and later `field.zip`) in `MyDrive/cropxai/raw/` (see `data/README.md`).
- Runtime → Change runtime type → GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Public repo; if you make it private, the clone needs a token.
REPO_URL = "https://github.com/ranjanvrma/crop-disease-xai.git"
BRANCH = "main"

import os
if not os.path.exists("/content/crop-disease-xai"):
    !git clone -b {BRANCH} {REPO_URL} /content/crop-disease-xai
else:
    !git -C /content/crop-disease-xai pull
%cd /content/crop-disease-xai
!git log --oneline -1

In [ ]:
!pip install -q -e ".[train,dev]"

In [ ]:
import os
from google.colab import userdata

DRIVE = "/content/drive/MyDrive/cropxai"
os.environ["CROPXAI_DATA"] = "/content/data"           # local disk: fast
os.environ["CROPXAI_CKPT"] = f"{DRIVE}/checkpoints"    # Drive: survives the session
os.makedirs(os.environ["CROPXAI_CKPT"], exist_ok=True)

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

In [ ]:
# Copy dataset zips from Drive to local disk and unzip (≈ a few minutes).
import os, subprocess
os.makedirs("/content/data", exist_ok=True)
for name in ["plantvillage", "plantdoc", "field"]:
    z = f"{DRIVE}/raw/{name}.zip"
    if os.path.exists(f"/content/data/{name}"):
        print(f"{name}: already unpacked")
    elif os.path.exists(z):
        subprocess.run(["unzip", "-q", z, "-d", "/content/data"], check=True)
        print(f"{name}: unpacked")
    else:
        print(f"{name}: no zip at {z} (field.zip is optional until you have photos)")

In [ ]:
# Use the split from the repo if committed, else the copy saved on Drive.
import os, shutil, glob
if not os.path.exists("splits/manifest_v1.csv") and glob.glob(f"{DRIVE}/splits/*"):
    for f in glob.glob(f"{DRIVE}/splits/*"):
        shutil.copy(f, "splits/")
    print("splits restored from Drive")
print("manifest present:", os.path.exists("splits/manifest_v1.csv"))

In [ ]:
# Sanity check: unit tests on synthetic data.
!python -m pytest -q

## Build the split (first time, or after adding field photos)
This writes `splits/manifest_v1.csv`. Commit the `splits/` files so the split is pinned.
The cell also copies them to Drive so you can pull them down and commit locally.

In [ ]:
!python scripts/prepare_data.py --config configs/experiment/prepare_data.yaml
!mkdir -p {DRIVE}/splits && cp splits/* {DRIVE}/splits/

## Phase 2: baselines (3 seeds each)
Each experiment trains seeds 0, 1, 2 and keeps the best-val checkpoint per seed on Drive.
`evaluate.py` is the only step that reads the field set. Run it once per finished experiment.

Suggested order (cheap first, so problems show early): `mnv3_fe`, `effnet_b0_fe`, `scratch_cnn`,
then the `_ft` runs, then `resnet50_*`, `effnet_b3_ft` (teacher), `deit_s_*`.
To run one seed while testing: add `--seed 0`.

In [ ]:
EXP = "mnv3_fe"   # any file name in configs/experiment/
!python scripts/train.py --config configs/experiment/{EXP}.yaml
!python scripts/evaluate.py --config configs/experiment/{EXP}.yaml

In [ ]:
# Results JSONs live in the repo checkout: keep a copy on Drive, then rebuild the tables.
!mkdir -p {DRIVE}/results && cp -r results/metrics {DRIVE}/results/
!python scripts/make_tables.py

## Phase 3: explanation quality (per trained seed)
`verify_closed_form.py` checks the gradient-free Grad-CAM++ against pytorch-grad-cam (should be ~1e-6 different).
`explain.py` computes faithfulness + localisation for Grad-CAM++, Grad-CAM and Score-CAM, and saves an example figure
to `results/figures/heatmaps/`. CNN experiments only (not `deit_s_*`).

In [ ]:
EXP, SEED = "mnv3_fe", 0
!python scripts/verify_closed_form.py --config configs/experiment/{EXP}.yaml --seed {SEED}
!python scripts/explain.py --config configs/experiment/{EXP}.yaml --seed {SEED}
!mkdir -p {DRIVE}/results && cp -r results/metrics results/figures {DRIVE}/results/

In [ ]:
# Look at the example heatmaps (lab test images vs. real field photos).
from IPython.display import Image, display
for split in ["test", "field_test"]:
    display(Image(f"results/figures/heatmaps/{EXP}_seed{SEED}_{split}.png", width=900))